### ReCoのTcデータを用いたクロスバリデーションによるLASSO回帰


**データ取得からデータ解析**

このファイルは
ReCoのTcデータを用いてLASSO回帰を行います。

LassoCVによりモデル性能評価をおこない、ハイパーパラメタを決める。
最適化したハイパーパラメタを用いて予測モデルを作り予測する。


In [ ]:
from sklearn.linear_model import LassoCV, RidgeCV
import warnings
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler, MaxAbsScaler
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline


pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 10)

# text用にwarningを消している。
warnings.filterwarnings('ignore')


基本的な部分は以下の

1. データ取得
2. データプリプロセス
3. データ解析

で済んでいます。
なお、KFoldのdefault parameterは今のところshuffle=FalseなのでshuffleするようにKFoldの使用を明示しています。


In [ ]:
# 結果を入れる辞書を容易する。
g_result = {}

In [ ]:
# "データ取得
def get_data(data_name: str):
    """データ取得

    Args:
        data_name (str): データ名
    """
    if data_name=="ReCo":
        df = pd.read_csv("../data/TC_ReCo_detail_descriptor.csv")
        descriptor_names = ['C_R', 'C_T', 'vol_per_atom', 'Z', 'f4', 'd5', 'L4f', 'S4f', 'J4f',
            '(g-1)J4f', '(2-g)J4f']
        # descriptor_names = ['C_R', 'vol_per_atom','Z']
        target_name = 'Tc'
    elif data_name=="ZBWE":
        df = pd.read_csv("../data/ZB_WZ_dE_rawdescriptor.csv")
        descriptor_names = ['IP_A', 'EA_A', 'EN_A', 'Highest_occ_A',
                            'Lowest_unocc_A', 'rs_A', 'rp_A', 'rd_A', 'IP_B', 'EA_B', 'EN_B',
                            'Highest_occ_B', 'Lowest_unocc_B', 'rs_B', 'rp_B', 'rd_B']
        target_name = 'dE'

    return df, descriptor_names, target_name

g_data_name = "ReCo" # ReCo, ZBWE
g_df, g_descriptor_names, g_target_name = get_data(g_data_name)

In [ ]:
!cat ../data/TC_ReCo_detail_descriptor.csv

内部でhyperparameterの最適化を行ってくれるLassoCVを用います。

In [ ]:
g_Xraw = g_df[g_descriptor_names].values
g_y = g_df[g_target_name].values

# データプリプロセス
g_scaler = StandardScaler()
g_scaler.fit(g_Xraw)
g_X = g_scaler.transform(g_Xraw)

# データ解析
g_kf = KFold(10, shuffle=True)
g_reg = LassoCV(cv=g_kf, alphas=np.logspace(-5, 2, 20))
g_reg.fit(g_X, g_y)
g_score = g_reg.score(g_X, g_y)
print("R2 score=", g_score)
g_result["LassoCV"]= {"R2": g_score}

g_yp = g_reg.predict(g_X)

scoreはLassoCVで作成した回帰モデルに対してR2 scoreを計算した値が入っています。
ypにはXに対して予測した値が入っている。
LassoCVを用いた回帰の説明はここまででです。

**可視化**

以下では可視化を用いて何を行ったかの説明を行います。
まず確認のためDataFrame dfの中身を表示します。

In [ ]:
g_df

規格化された説明変数を表示します。

In [ ]:
def show_X(X):
    """説明変数の図示

    Args:
        X (np.ndarray): 説明変数
    """
    fig, ax = plt.subplots()
    plt.plot(X, ".-")
    plt.show()
show_X(g_X)

目的変数を表示します。

In [ ]:
def show_hist(y):
    """目的変数のhistogram図示

    Args:
        y (np.ndarray): 目的変数
    """
    fig, ax = plt.subplots()
    ax.hist(y)
    ax.set_xlabel("y")
    fig.show()
show_hist(g_y)

LassoCVが選択したハイパーパラメタは以下です。

In [ ]:
g_reg.alpha_, np.log10(g_reg.alpha_)

回帰モデルの係数を示します。

In [ ]:
print(g_descriptor_names)
print(g_reg.coef_, g_reg.intercept_)

目的変数とその予測値を表示します。

In [ ]:
def show_y_yp(y,yp):
    """目的変数、観測値と予測値の図示

    Args:
        y (np.ndarray): 目的変数観測値
        yp (np.ndarray): 目的変数予測値
    """
    fig, ax =plt.subplots(figsize=(5, 5))
    ax.plot(y, yp, "o")
    yall = np.hstack([yp, y])
    y1, y2 = np.min(yall), np.max(yall)
    ax.plot([y1, y2], [y1, y2], "--")  # 対角線を引く
    ax.set_xlabel("$y_{obs}$")
    ax.set_ylabel("$y_{pred}$")
    fig.show()
    
show_y_yp(g_y,g_yp)

### 付録１



変数alphas_,
mse_path_
に評価したハイパーパラメタの値のその時のCV MSE値が入っています。
scikit learnでは平均値しか見ていませんが、標準偏差を含めて図示します。


In [ ]:
from scipy.stats import pearsonr


def show_lassocv_path(reg):
    """show lasso CV MSE.

    Args:
        reg (regressor): regressor
    """
    mse_mean = np.mean(reg.mse_path_, axis=-1)
    mse_std = np.std(reg.mse_path_, axis=-1)
    mse_mean_min = np.min(mse_mean)

    fig, ax = plt.subplots()

    ax.set_xlabel("log10(alpha)")
    ax.set_ylabel("score")
    # ax.set_ylim((0,30000))
    # plt.loglog(reg.alphas_,mse_mean,"o-")
    ax.set_xscale("log")
    ax.errorbar(reg.alphas_, mse_mean, yerr=mse_std, capsize=5, fmt="o")
    ax.axhline(mse_mean_min, linestyle="--")  # 縦線を引く
    ax.axvline(np.log10(reg.alpha_), linestyle='--')  # 横線を引く
    plt.show()

show_lassocv_path(g_reg)


別手法として
DataFrameの箱ひげ図(boxplot)表示を用います。
外れ値が存在することが分かります。
また標準偏差はやや大きいことと、MSE平均値の最小log10(alpha)〜-0.58にあることが分かります。

In [ ]:
def boxplot_lassocv_path(reg):
    """show lassoCV MSE in the box plot.

    Args:
        reg (regressor): regressor
    """
    iorder = np.argsort(reg.alphas_)
    label = []
    for x in np.log10(reg.alphas_[iorder]):
        label.append("{:.2f}".format(x))
    df_mse_path = pd.DataFrame(reg.mse_path_[iorder, :].T, columns=label)
    
    fig, ax = plt.subplots()
    df_mse_path.boxplot(rot=90, ax=ax)
    #plt.ylim( (0,40000))
    ax.set_xlabel("log10(alpha)")
    fig.savefig("image_executed/LassoCV_boxplot.png")
    fig.show()

boxplot_lassocv_path(g_reg)


### 付録２

LassoCVが何を行っているか見ていきます。

reg.alpha_を用いて別にLassoを作り(X,y)でfitして同じ回帰モデルであることを示します。

In [ ]:
from sklearn.linear_model import Lasso


def make_lasso(X, y, yp, alpha):
    """evaluate score using lasso

    Args:
        X (np.array): descriptor
        y (np.array): target variable
        yp (np.array): predicted target variable
        alpha (float): hyperparameter of Ridge regression
    """
    reglasso = Lasso(alpha=alpha, fit_intercept=True)
    reglasso.fit(X, y)
    yp2 = reglasso.predict(X)

    corr = pearsonr(yp, yp2)
    print("Pearson correlation=", corr)

    score = reglasso.score(X, y)
    print("R2 score=", score)
    
    return {"R2": score}

g_result["Lasso_opt"] = make_lasso(g_X, g_y, g_yp, g_reg.alpha_)


全ての結果を表示する。

In [ ]:
g_dfresult = pd.DataFrame(g_result)
g_dfresult["diff"] = g_dfresult["LassoCV"] - g_dfresult["Lasso_opt"]
g_dfresult

R2 scoreも全く同じ値です。

LassoCVは

1. CVを行う。
    1. テストデータからMSEを出す。
    2. MSEが最も小さいalpha(最適なalpha)を求める。
2. 最適なalphaを用いて全データを用いて回帰モデルを作りなおす。

ということをしてます。

**CV (test)の再解析と可視化**

最後にCV testの回帰スコアを出力します。

In [ ]:
from sklearn.metrics import r2_score
from sklearn.linear_model import Lasso
from sklearn.model_selection import KFold


def linear_regression_CV_score_ytestp(X, y, alpha, n_splits=10):
    """make prediction with cross validated lasso 

    Args:
        X (np.array): descriptor
        y (np.array): target variable
        alpha (float): hyperparameter
        n_splits (int, optional): the number of splits of the CV. Defaults to 10.

    Returns:
        dict: the mean value of the R2 score,  the stddev vlaue of the R2 score, 
              a list of y_test, a list of y_test^predict
    """
    reg = Lasso(alpha=alpha)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=1)

    test_score_list = []
    ytest_list = []
    ytestp_list = []
    for train, test in kf.split(X):
        Xtrain, ytrain = X[train], y[train]
        Xtest, ytest = X[test], y[test]
        reg.fit(Xtrain, ytrain)

        ytestp = reg.predict(Xtest)
        ytest_list.extend(ytest)
        ytestp_list.extend(ytestp)

        test_score = r2_score(ytest, ytestp)
        test_score_list.append(test_score)

    return {"R2":np.mean(test_score_list), 
            "R2.std": np.std(test_score_list), 
            "ytest": ytest_list, "ytestp":ytestp_list}


g_result_ = linear_regression_CV_score_ytestp(
    g_X, g_y, g_reg.alpha_)

g_test_score_mean, g_test_score_std, g_ytest_list, g_ytestp_list = g_result_["R2"], g_result_["R2.std"], \
                        g_result_["ytest"], g_result_["ytestp"]

print("R2 CV(test)={}({})".format(g_test_score_mean, g_test_score_std))


CV(test)のR2はデータ全体からモデルを作成したR2よりも小さいことが普通です。

CV testの観測値がresult_["ytest"]、
CV testの予測値がresult["ytestp"]
に入っています。これを図示します。

In [ ]:
import copy
def show_y_yp(ytest_list, ytestp_list):
    """show y and y^predict

    Args:
        ytest_list (list): y_test
        ytestp_list (lsit): y_test^predict
    """
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(ytest_list, ytestp_list, "o")
    
    # make min. and max. for an aux. line.
    yall = copy.deepcopy(ytest_list) # don't touch ytest_list
    yall.extend(ytestp_list) # make a list of all values
    ymin = np.min(yall)
    ymax = np.max(yall)
    ax.plot((ymin,ymax),(ymin,ymax), "--")
    
    ax.set_xlabel("$y_{expr}$")
    ax.set_ylabel("$y_{pred}$ (CV test)")
    plt.savefig("image_executed/LassoCV_y_yCVtest.png")
    plt.show()

show_y_yp(g_result_["ytest"], g_result_["ytestp"])


##### 問題
他のデータでこのscriptを動かす。